# Data Quality Assessment

Basic quality checks for the Kaggle Shelter Animal Outcomes dataset.

In [24]:
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

In [25]:
from configurations import (
    OUTPUT_DIR,
    RANDOM_STATE,
    TARGET,
    TEST_PATH,
    TRAIN_PATH,
)

# Filename for the missing-value figure.
MISSING_VALUES_FILENAME = "data_quality_missing_values.png"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"Train: {train.shape}, Test: {test.shape}")

Train: (26729, 10), Test: (11456, 8)


In [26]:
import importlib
import plotting

importlib.reload(plotting)
from plotting import show_and_save

In [27]:
def quality_report(frame):
    """Summarize column types, missing values, and unique values."""
    report = pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "missing": frame.isna().sum(),
        "missing_pct": frame.isna().mean().mul(100).round(2),
        "unique": frame.nunique(dropna=False),
    })
    return report.sort_values("missing_pct", ascending=False)


def parse_age_in_months(value):
    """Convert an age string to months."""
    if pd.isna(value):
        return np.nan
    amount, unit = str(value).split(maxsplit=1)
    multiplier = {"day": 1 / 30, "days": 1 / 30, "week": 1 / 4.345, "weeks": 1 / 4.345,
                  "month": 1, "months": 1, "year": 12, "years": 12}
    return float(amount) * multiplier[unit.lower()]


def categorical_mismatch(train_frame, test_frame):
    """Find categories that occur in only one dataset."""
    columns = train_frame.columns.intersection(test_frame.columns)
    return {
        column: sorted(set(train_frame[column].dropna().unique()) ^ set(test_frame[column].dropna().unique()))
        for column in columns
        if train_frame[column].dtype == "object"
    }

In [28]:
train_quality = quality_report(train)
test_quality = quality_report(test)

print("Train quality report")
display(train_quality)
print("Test quality report")
display(test_quality)

print(f"Train duplicate rows: {train.duplicated().sum()}")
print(f"Test duplicate rows: {test.duplicated().sum()}")
print("Train/test category differences:")
display(pd.Series(categorical_mismatch(train, test), name="only_one_dataset"))

Train quality report


,dtype,missing,missing_pct,unique
OutcomeSubtype,str,13612,50.93,17
Name,str,7691,28.77,6375
AgeuponOutcome,str,18,0.07,45
AnimalID,str,0,0.00,26729
OutcomeType,str,0,0.00,5
DateTime,str,0,0.00,22918
AnimalType,str,0,0.00,2
SexuponOutcome,str,1,0.00,6
Breed,str,0,0.00,1380
Color,str,0,0.00,366


Test quality report


,dtype,missing,missing_pct,unique
Name,str,3225,28.15,3713
AgeuponOutcome,str,6,0.05,46
DateTime,str,0,0.00,10575
ID,int64,0,0.00,11456
AnimalType,str,0,0.00,2
SexuponOutcome,str,0,0.00,5
Breed,str,0,0.00,913
Color,str,0,0.00,277


Train duplicate rows: 0
Test duplicate rows: 0
Train/test category differences:


Series([], Name: only_one_dataset, dtype: object)

In [29]:
train_checked = train.copy()
train_checked["DateTime"] = pd.to_datetime(train_checked["DateTime"], errors="coerce")
train_checked["AgeMonths"] = train_checked["AgeuponOutcome"].map(parse_age_in_months)

invalid_datetime = train_checked["DateTime"].isna().sum()
invalid_age = train_checked["AgeMonths"].isna().sum()
print(f"Invalid datetimes: {invalid_datetime}")
print(f"Invalid or missing ages: {invalid_age}")
display(train_checked["AgeMonths"].describe().to_frame("value"))

Invalid datetimes: 0
Invalid or missing ages: 18


,value
count,26711.000000
mean,26.126838
std,35.589263
min,0.000000
25%,2.000000
50%,12.000000
75%,36.000000
max,240.000000


In [30]:
missing = train_quality.query("missing > 0").sort_values("missing", ascending=True)
fig = px.bar(
    missing,
    x="missing",
    y=missing.index,
    orientation="h",
    text="missing_pct",
    labels={"y": "Column", "missing": "Missing values", "missing_pct": "Missing (%)"},
    title="Missing values in the training dataset",
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(uniformtext_minsize=8, uniformtext_mode="hide")
show_and_save(fig, MISSING_VALUES_FILENAME)

Figure saved: output/data_quality_missing_values.png


PosixPath('output/data_quality_missing_values.png')